In [ ]:
# Facebook AI Similarity Search
!pip install faiss

In [1]:
# Start by setting the load location to Naomi's root
import os
import sys

sys.path.insert(0, os.path.expanduser('~/Naomi'))


# Convert a word ("word") to a keyword ("{word}")
def to_keyword(word):
    return "{}{}{}".format("{", word, "}")

In [2]:
# Next, load all the intents from Naomi
import configparser
import importlib
import inspect
from naomi import i18n
from naomi import profile
from pprint import pprint

class PluginInfo(object):
    def __init__(self, cp, plugin_class, translations, directory):
        self._cp = cp
        self._plugin_class = plugin_class
        self._translations = translations
        self._path = directory

    def _get_optional_info(self, *args):
        try:
            value = self._cp.get(*args)
        except configparser.Error:
            value = ''
        return value

    @property
    def plugin_class(self):
        return self._plugin_class

    @plugin_class.setter
    def plugin_class(self, value):
        if self._plugin_class is not None:
            raise RuntimeError('Changing a plugin class is not allowed!')
        self._plugins_class = value

    @property
    def translations(self):
        return self._translations

    @property
    def name(self):
        return self._cp.get('Plugin', 'Name')

    @property
    def version(self):
        return self._cp.get('Plugin', 'Version')

    @property
    def license(self):
        return self._cp.get('Plugin', 'License')

    @property
    def description(self):
        return self._get_optional_info('Plugin', 'Description')

    @property
    def url(self):
        return self._get_optional_info('Plugin', 'URL')

    @property
    def author_name(self):
        return self._get_optional_info('Author', 'Name')

    @property
    def author_email(self):
        return self._get_optional_info('Author', 'Email')

    @property
    def author_url(self):
        return self._get_optional_info('Author', 'URL')

cp = configparser.RawConfigParser()
plugin_dirs = ["/home/jess/Naomi/plugins/speechhandler", "/home/jess/.config/naomi/plugins/speechhandler"]
plugins = {}
intent_templates = dict()
for plugin_dir in plugin_dirs:
    for root, dirs, files in os.walk(plugin_dir, topdown=True):
        for name in files:
            if name == 'plugin.info':
                sys.path.append(root)
                cp.read(os.path.join(root, name))
                plugin_name = cp.get('Plugin', 'Name')
                # print(f"{root}\t{plugin_name}")
                spec = importlib.util.spec_from_file_location(
                    plugin_name,
                    os.path.join(root, '__init__.py')
                )
                print(root)
                mod = importlib.util.module_from_spec(spec)
                sys.modules[mod.__package__] = mod
                spec.loader.exec_module(mod)
                plugin_classes = inspect.getmembers(
                    mod,
                    lambda cls: inspect.isclass(cls)
                )
                for plugin_class in plugin_classes:
                    info = PluginInfo(
                        cp,
                        plugin_class[1],
                        i18n.parse_translations('/home/jess/Naomi/naomi/data/locale'),
                        root
                    )
                    try:
                        plugin = plugin_class[1](info)
                        intents = plugin.intents()
                        for intent in intents:
                            intent_templates[intent]=[]
                            for locale in intents[intent]['locale']:
                                if locale == 'en-US':
                                    templates = intents[intent]['locale'][locale]['templates']
                                    if 'keywords' in intents[intent]['locale'][locale]:
                                        for keyword in intents[intent]['locale'][locale]['keywords']:
                                            for template in templates:
                                                if to_keyword(keyword) in template:
                                                    templates.extend([template.replace(to_keyword(keyword), word.upper()) for word in intents[intent]['locale'][locale]['keywords'][keyword]])
                                    intent_templates[intent].extend(templates)
                    except Exception as e:
                        print(f"Error: {e}")
pprint(intent_templates)

Connection error while trying to access server localhost:6600: Connection refused (Errno: 111)


/home/jess/Naomi/plugins/speechhandler/wwis_weather
Translations: {'en-US': <gettext.NullTranslations object at 0x7f72684cf950>}
/home/jess/Naomi/plugins/speechhandler/clock
Translations: {'en-US': <gettext.NullTranslations object at 0x7f726830f910>}
/home/jess/Naomi/plugins/speechhandler/shutdownplugin
Translations: {'en-US': <gettext.NullTranslations object at 0x7f726830fdd0>}
/home/jess/Naomi/plugins/speechhandler/news
Translations: {'en-US': <gettext.NullTranslations object at 0x7f7268349210>}
/home/jess/Naomi/plugins/speechhandler/check_email
Translations: {'en-US': <gettext.NullTranslations object at 0x7f726834a110>}
/home/jess/Naomi/plugins/speechhandler/mpdcontrol
Translations: {'en-US': <gettext.NullTranslations object at 0x7f726834b650>}
Error: Connection failed
/home/jess/Naomi/plugins/speechhandler/life
Translations: {'en-US': <gettext.NullTranslations object at 0x7f726834a250>}
/home/jess/Naomi/plugins/speechhandler/stop
Translations: {'en-US': <gettext.NullTranslations ob

                      'WHAT IS THE WEATHER FORECAST FOR SAN FRANCISCO ON '
                      'FRIDAY AFTERNOON',
                      'WHAT IS THE WEATHER FORECAST FOR SAN FRANCISCO ON '
                      'SATURDAY AFTERNOON',
                      'WHAT IS THE WEATHER FORECAST FOR SAN FRANCISCO ON TODAY '
                      'EVENING',
                      'WHAT IS THE WEATHER FORECAST FOR SAN FRANCISCO ON '
                      'TOMORROW EVENING',
                      'WHAT IS THE WEATHER FORECAST FOR SAN FRANCISCO ON '
                      'SUNDAY EVENING',
                      'WHAT IS THE WEATHER FORECAST FOR SAN FRANCISCO ON '
                      'MONDAY EVENING',
                      'WHAT IS THE WEATHER FORECAST FOR SAN FRANCISCO ON '
                      'TUESDAY EVENING',
                      'WHAT IS THE WEATHER FORECAST FOR SAN FRANCISCO ON '
                      'WEDNESDAY EVENING',
                      'WHAT IS THE WEATHER FORECAST FOR SAN FRANCISCO

In [3]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# Prepare the dataset and metadata
texts = []
intents = []

for intent, phrases in intent_templates.items():
    for phrase in phrases:
        texts.append(phrase)
        intents.append(intent)

# Load embedding model
model = SentenceTransformer("sentence-transformers/nli-mpnet-base-v2")

# Compute embeddings
embeddings = model.encode(texts, convert_to_numpy=True, normalize_embeddings=True)
print(embeddings)

# Create FAISS index (cosine = inner product on normalized vectors)
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

# Search function
def detect_intent(query, top_k=1):
    query_vector = model.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    scores, indices = index.search(query_vector, top_k)

    results = []
    for idx, score in zip(indices[0], scores[0]):
        results.append({
            "text": texts[idx],
            "intent": intents[idx],
            "score": float(score)
        })
    return results

2025-07-07 10:42:30.735498: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1751899350.755440  526228 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1751899350.760918  526228 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-07-07 10:42:30.781092: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


[[-1.87536003e-03 -6.51287884e-02  2.36912202e-02 ...  1.13148279e-02
  -7.72576965e-03  5.01537416e-03]
 [-4.43227589e-03 -6.64739385e-02  2.15001591e-02 ...  5.72549319e-03
  -2.81822309e-02 -1.01438081e-02]
 [-8.43878090e-03 -4.51899655e-02  2.60083061e-02 ...  1.30947083e-02
  -7.77546130e-03  1.81935681e-03]
 ...
 [ 9.49552003e-03 -1.07752644e-01 -2.62162536e-02 ...  2.00865790e-02
  -1.86415315e-02  5.93831297e-03]
 [-2.88311625e-03 -1.13439657e-01 -7.24473828e-03 ...  1.77199878e-02
  -3.48682925e-02 -4.61524651e-05]
 [ 1.35305235e-02 -8.86806026e-02 -2.42390260e-02 ...  1.95186306e-02
  -1.44886384e-02  1.59479994e-02]]


In [29]:
query = "what's the time?"
results = detect_intent(query)

for result in results:
    print(result) 

{'text': 'WHAT IS THE TIME', 'intent': 'ClockIntent', 'score': 0.9485380053520203}


In [28]:
# Load transcriptions and verified intents from audiolog.db
import sqlite3

conn = sqlite3.connect(os.path.expanduser("~/Projects/Speaker Recognition/audiolog.db"))
c = conn.cursor()
sql = " ".join([
    "select",
        "iif(verified_transcription='',transcription,verified_transcription) as transcription,",
        "verified_intent",
    "from audiolog",
    "where verified_intent > ''",
        "and (verified_transcription > '' or transcription > '')"
])
res = c.execute(sql)
row = res.fetchone()
while row:
    # print(row)
    # Find out what intent FAISS would choose
    result = detect_intent(row[0])
    intent = result[0]['intent']
    score = result[0]['score']
    if score < 0.5:
        intent = "unclear"
    if row[1] != intent:
        print(f"text: {row[0]}, actual intent: {row[1]}, detected intent: {intent}, score: {score}" )
        print()
    row = res.fetchone()
conn.close()

text: WHEN ON I OFF WILL FOR OFF, actual intent: unclear, detected intent: ShutdownIntent, score: 0.579043984413147

text: I READ YOU MY SAN, actual intent: unclear, detected intent: NewsIntent, score: 0.5683563947677612

text: BE MORNING YOU IT, actual intent: unclear, detected intent: GreetingsIntent, score: 0.8085642457008362

text: BE TURN LAUGH I THERE AND YOU THIS, actual intent: unclear, detected intent: JokeIntent, score: 0.7666413187980652

text: BE WHEN A I NOW FRIENDS I UP FRIDAY TURN AND WHOSE IT BEING THERE AND I WHEN LAUGH THERE, actual intent: unclear, detected intent: JokeIntent, score: 0.5813553333282471

text: THIS I TOMORROW SLEET WORK I I WORK TODAY I FOR I I, actual intent: unclear, detected intent: WeatherIntent, score: 0.6992573738098145

text: NOW NAOMI IT IT WORK NAOMI IS NOW, actual intent: HackerNewsIntent, detected intent: unclear, score: 0.3054477572441101

text: WHAT NAOMI WHAT ON ON, actual intent: WeatherIntent, detected intent: unclear, score: 0.4696625

text: BE FOR HAVE, actual intent: unclear, detected intent: ThankYouIntent, score: 0.5758096575737

text: A MONDAY YOU TODAY I AND ON UNIVERSE YOU WORK KNOW, actual intent: unclear, detected intent: WeatherIntent, score: 0.5870964527130127

text: BE IT NEWS, actual intent: unclear, detected intent: NewsIntent, score: 0.7094486951828003

text: BE A WILL I KNOW MY SUNNY, actual intent: unclear, detected intent: WeatherIntent, score: 0.6391788721084595

text: BE TELL THIS YOU HACKER, actual intent: unclear, detected intent: HackerNewsIntent, score: 0.732269287109375

text: BE ON RAIN WEATHER READ WORK AFTERNOON FOR, actual intent: unclear, detected intent: WeatherIntent, score: 0.7125563621520996

text: A SUNNY I NEWS OF, actual intent: unclear, detected intent: WeatherIntent, score: 0.677685022354126

text: BE HAPPENING TOMORROW TOMORROW SAN WHEN HAVE, actual intent: unclear, detected intent: WeatherIntent, score: 0.6717333793640137

text: BE I LAUGH FRIENDS WHOSE KNOW RAINING WILL, actu

text: BE AND YOU SAN LAUGH WORK THURSDAY MORNING, actual intent: unclear, detected intent: WeatherIntent, score: 0.6056501865386963

text: BE SEATTLE OFF SHUT BE A WHEN TURN, actual intent: unclear, detected intent: ShutdownIntent, score: 0.5296337604522705

text: NEWS YOU NOW MEANING IS, actual intent: unclear, detected intent: NewsIntent, score: 0.7196874022483826

text: TURN WHOSE, actual intent: unclear, detected intent: LEDIntent, score: 0.5036492347717285

text: BE, actual intent: unclear, detected intent: GreetingsIntent, score: 0.6217020750045776

text: BE TURN, actual intent: unclear, detected intent: LEDIntent, score: 0.5023535490036011

text: NAOMI QUELLE HEURE EST-IL, actual intent: ClockIntent, detected intent: unclear, score: 0.3656392991542816

text: NAOMI QUELLE HEURE EST-IL, actual intent: ClockIntent, detected intent: unclear, score: 0.3656392991542816

text: TITRES ONT, actual intent: unclear, detected intent: GreetingsIntent, score: 0.5037766695022583

text: THANK Y

text: HOW ARE YOU MAGICVOICE, actual intent: HowAreYouIntent, detected intent: HowAreYou, score: 0.59347003698349

text: HOW ARE YOU TODAY MAGICVOICE, actual intent: HowAreYouIntent, detected intent: HowAreYou, score: 0.5931004285812378

text: WHO'S THERE, actual intent: unclear, detected intent: CheckNetworkIntent, score: 0.5010098814964294

text: MAGICVOICE HOW ARE YOU TODAY, actual intent: HowAreYouIntent, detected intent: HowAreYou, score: 0.5970054268836975

text: THANK YOU MAGICVOICE, actual intent: ThankYouIntent, detected intent: unclear, score: 0.49707555770874023

text: THANK YOU MAGICVOICE, actual intent: ThankYouIntent, detected intent: unclear, score: 0.49707555770874023

text: HOW ARE YOU MAGICVOICE, actual intent: HowAreYouIntent, detected intent: HowAreYou, score: 0.59347003698349

text: MAGICVOICE HOW ARE YOU, actual intent: HowAreYouIntent, detected intent: HowAreYou, score: 0.65506911277771

text: MAGICVOICE HOW YOU DOING TODAY, actual intent: HowAreYouIntent, detect

text: NO THANK YOU, actual intent: unclear, detected intent: ThankYouIntent, score: 0.5438997149467468

text: NO THANK YOU, actual intent: unclear, detected intent: ThankYouIntent, score: 0.5438997149467468

text: THANK YOU MAGICVOICE, actual intent: ThankYouIntent, detected intent: unclear, score: 0.49707555770874023

text: GOOD MORNING MAGICVOICE, actual intent: HowAreYouIntent, detected intent: GreetingsIntent, score: 0.6434599757194519

text: NO THANK YOU, actual intent: unclear, detected intent: ThankYouIntent, score: 0.5438997149467468

text: MAGICVOICE HOW ARE YOU TODAY, actual intent: HowAreYouIntent, detected intent: HowAreYou, score: 0.5970054268836975

text: MAGICVOICE HOW ARE YOU, actual intent: HowAreYouIntent, detected intent: HowAreYou, score: 0.65506911277771

text: NO THANK YOU, actual intent: unclear, detected intent: ThankYouIntent, score: 0.5438997149467468

text: GOOD MORNING MAGICVOICE, actual intent: HowAreYouIntent, detected intent: GreetingsIntent, score: 0.643

...

text: WHO'S THERE, actual intent: unclear, detected intent: CheckNetworkIntent, score: 0.5010098814964294

text: GOOD MORNING MAGICVOICE, actual intent: unclear, detected intent: GreetingsIntent, score: 0.6434599757194519

text: GOOD MORNING MAGICVOICE, actual intent: HowAreYouIntent, detected intent: GreetingsIntent, score: 0.6434599757194519

text: THANK YOU MAGICVOICE, actual intent: ThankYouIntent, detected intent: unclear, score: 0.49707555770874023

text: THANK YOU MAGICVOICE, actual intent: ThankYouIntent, detected intent: unclear, score: 0.49707555770874023

text: THANK YOU MAGICVOICE, actual intent: ThankYouIntent, detected intent: unclear, score: 0.49707555770874023

text: GOOD MORNING MAGICVOICE, actual intent: HowAreYouIntent, detected intent: GreetingsIntent, score: 0.6434599757194519

text: MAGICVOICE HOW ARE YOU, actual intent: HowAreYouIntent, detected intent: HowAreYou, score: 0.65506911277771

text: MAGICVOICE PLAY HADESTOWN, actual intent: MPDControlIntent, detected 

text: GOOD MORNING MAGICVOICE, actual intent: HowAreYouIntent, detected intent: GreetingsIntent, score: 0.6434599757194519

text: THANK YOU MAGICVOICE, actual intent: ThankYouIntent, detected intent: unclear, score: 0.49707555770874023

text: THANK YOU MAGICVOICE, actual intent: ThankYouIntent, detected intent: unclear, score: 0.49707555770874023

text: TO THE TIME IS ARE QUESTION BE, actual intent: unclear, detected intent: ClockIntent, score: 0.7825425863265991

text: THANK YOU MAGICVOICE, actual intent: ThankYouIntent, detected intent: unclear, score: 0.49707555770874023

text: SAN SLEETING IN THE TURN ARE RAIN, actual intent: unclear, detected intent: WeatherIntent, score: 0.7057138085365295

text: ARE THERE ARE A IT OF EMAILS WHAT WHEN A AND FOR WHEN YOU AND, actual intent: unclear, detected intent: CheckEmailIntent, score: 0.5824763178825378

text: FOR READ, actual intent: unclear, detected intent: NewsIntent, score: 0.5919423699378967

text: IN A THE SUNNY STOP AND IT THERE IS D

text: THANK YOU MAGICVOICE, actual intent: ThankYouIntent, detected intent: unclear, score: 0.49707555770874023

text: GOOD MORNING MAGICVOICE, actual intent: HowAreYouIntent, detected intent: GreetingsIntent, score: 0.6434599757194519

text: WHAT IS IN, actual intent: unclear, detected intent: NewsIntent, score: 0.5204429626464844

text: IS IT, actual intent: unclear, detected intent: ClockIntent, score: 0.5294951796531677

text: THANK YOU MAGICVOICE, actual intent: ThankYouIntent, detected intent: unclear, score: 0.49707555770874023

text: THANK YOU MAGICVOICE, actual intent: ThankYouIntent, detected intent: unclear, score: 0.49707555770874023

text: GOOD, actual intent: unclear, detected intent: ThankYouIntent, score: 0.6337916851043701

text: THANK YOU MAGICVOICE, actual intent: ThankYouIntent, detected intent: unclear, score: 0.49707555770874023

text: GOOD MORNING MAGICVOICE, actual intent: HowAreYouIntent, detected intent: GreetingsIntent, score: 0.6434599757194519

text: ARE TH

text: MAGICVOICE WHEN IS HALLOWEEN THIS YEAR, actual intent: DateIntent, detected intent: unclear, score: 0.4595644474029541

text: GOOD MORNING MAGICVOICE, actual intent: HowAreYouIntent, detected intent: GreetingsIntent, score: 0.6434599757194519

text: TO THE ABOUT BE IT THE FRIENDS FOR THE THIS IN IS, actual intent: unclear, detected intent: CheckNetworkIntent, score: 0.5368142127990723

text: TO MAKE ARE THE THE NOW ON I THE I ARE BUT A ALL YOU KNOW IT A ABOUT UP THE IN FOR TO THE THE WINDY IT NOW ITALY READ IT, actual intent: unclear, detected intent: WeatherIntent, score: 0.5057784914970398

text: OKAY YEAH, actual intent: unclear, detected intent: GreetingsIntent, score: 0.6319544911384583

text: THANK YOU MAGICVOICE, actual intent: ThankYouIntent, detected intent: unclear, score: 0.49707555770874023

text: MAGICVOICE EXIT, actual intent: StopIntent, detected intent: GoodbyeIntent, score: 0.5187097787857056

text: THANK YOU MAGICVOICE, actual intent: ThankYouIntent, detected in

text: GOOD MORNING MAGICVOICE, actual intent: HowAreYouIntent, detected intent: GreetingsIntent, score: 0.6434599757194519

text: MAGICVOICE HOW ARE YOU, actual intent: HowAreYouIntent, detected intent: HowAreYou, score: 0.65506911277771

text: HOW ARE YOU MAGICVOICE, actual intent: HowAreYouIntent, detected intent: HowAreYou, score: 0.59347003698349

text: THANK YOU MAGICVOICE, actual intent: ThankYouIntent, detected intent: unclear, score: 0.49707555770874023

text: THANK YOU MAGICVOICE, actual intent: ThankYouIntent, detected intent: unclear, score: 0.49707555770874023

text: THANK YOU MAGICVOICE, actual intent: ThankYouIntent, detected intent: unclear, score: 0.49707555770874023

text: THANK YOU MAGICVOICE, actual intent: ThankYouIntent, detected intent: unclear, score: 0.49707555770874023

text: THANK YOU MAGICVOICE, actual intent: ThankYouIntent, detected intent: unclear, score: 0.49707555770874023

text: THANK YOU MAGICVOICE, actual intent: ThankYouIntent, detected intent: uncle